# Music Lab — ACE-Step 1.5 en Colab T4

Levanta `services/music-lab` con el motor real **ACE-Step 1.5** y publica un
túnel `*.trycloudflare.com` para conectar la API Node (`MUSIC_LAB_URL`).

**Requisitos**
- Runtime con GPU: *Runtime → Change runtime type → T4 GPU*.
- El DiT-only (`ACE_STEP_USE_LM=false`) y `ACE_STEP_OFFLOAD=true` son los
  defaults del script para entrar en los 16 GB de la T4.
- La primera generación descarga ~8 GB de checkpoints desde Hugging Face.

> Notebook pensado para pruebas. No expongas el túnel sin
> `MUSIC_LAB_API_KEY`.

In [ ]:
!nvidia-smi
!python3 --version

In [ ]:
REPO_URL = "https://github.com/<org>/son1kverse-unified.git"  # @param {type:"string"}
REPO_DIR = "/content/son1kverse-unified"

import os
if not os.path.isdir(REPO_DIR):
    if "<org>" in REPO_URL:
        raise ValueError("Define REPO_URL con el remoto real del monorepo")
    !git clone $REPO_URL $REPO_DIR

%cd {REPO_DIR}/services/music-lab

In [ ]:
import getpass, os

api_key = getpass.getpass("MUSIC_LAB_API_KEY (vacío = sin auth): ")
os.environ["MUSIC_LAB_API_KEY"] = api_key
os.environ["ACE_STEP_OFFLOAD"] = "true"
os.environ["ACE_STEP_USE_LM"] = "false"
os.environ["ACE_STEP_DEVICE"] = "cuda"

!nohup bash scripts/colab_run.sh >/content/colab_run.log 2>&1 &
print("Arrancando Music Lab en segundo plano…")

In [ ]:
import pathlib, re, time

log = pathlib.Path("/content/colab_run.log")
public_url = None
for attempt in range(120):
    text = log.read_text(errors="ignore") if log.exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        public_url = match.group(0)
        break
    if attempt % 6 == 0:
        print(f"Esperando instalación/arranque… ({attempt * 5}s)")
    time.sleep(5)

if public_url is None:
    print("No se encontró la URL. Últimas líneas del log:")
    print("\n".join(text.splitlines()[-40:]))
    raise RuntimeError("Revisa /content/colab_run.log")

print("Music Lab URL:", public_url)

In [ ]:
!curl -fsS {public_url}/health | python3 -m json.tool

## Conectar la API Node

En un host con PostgreSQL y el monorepo:

```bash
MUSIC_LAB_URL=<public_url> \
MUSIC_LAB_API_KEY=<la misma clave> \
make dev
```

Smoke real (desde la raíz del repo, con la API en `:3001`):

```bash
E2E_ENGINE=ace-step E2E_DURATION_S=30 node apps/api/scripts/e2e-audio.mjs
```

La primera generación descarga los checkpoints y puede tardar bastante;
el script espera hasta 10 minutos. El reporte muestra `engine`, `mode`,
`metadata.ham`, `metadata.mastering` y `metadata.compute`.

Para detener el túnel y uvicorn: *Runtime → Disconnect and delete runtime*.